## 添加消息历史记录（内存）

RunnableWithMessageHistory 允许我们将消息历史记录添加到某些类型的链中。它包装另一个 Runnable 并管理它的聊天消息历史记录。

具体来说，它可用于任何将以下之一作为输入的 Runnable

- 一个字典，其键采用 BaseMessage 序列
- 一个字典，其键将最新消息作为 BaseMessage 的字符串或序列，以及一个单独的键，将历史消息作为字符串或序列

并作为输出之一返回

- 可以视为 AIMessage 内容的字符串
- 一个字典，其键包含 BaseMessage 序列

让我们看一些示例，看看它是如何工作的。首先我们构造一个可运行的程序（这里接受一个字典作为输入并返回一条消息作为输出）：




In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一位擅长{ability}的助手。 用 20 个字或更少的字数进行简洁概要的回复",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
runnable = prompt | model

为了管理消息历史记录，我们需要： 
1. 这个可运行程序； 
2. 返回 BaseChatMessageHistory 实例的可调用函数。

查看内存集成页面，了解使用 Redis 和其他提供程序实现聊天消息历史记录。在这里，我们演示如何使用内存中 ChatMessageHistory 以及使用 RedisChatMessageHistory 进行更持久的存储。

## In-memory 内存中

下面我们展示了一个简单的示例，其中聊天历史记录位于内存中，在本例中通过全局 Python 字典。

我们构造一个可调用的 get_session_history ，它引用此字典以返回 ChatMessageHistory 的实例。可以通过在运行时将配置传递给 RunnableWithMessageHistory 来指定可调用的参数。默认情况下，配置参数应为单个字符串 session_id 。这可以通过 history_factory_config kwarg 进行调整。

使用单参数默认值：

In [2]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

请注意，我们指定了 input_messages_key （被视为最新输入消息的键）和 history_messages_key （添加历史消息的键）。

当调用这个新的可运行时，我们通过配置参数指定相应的聊天历史记录：

In [3]:
with_message_history.invoke(
    {"ability": "数学", "input": "余弦是什么意思？"},
    config={"configurable": {"session_id": "abc123"}},
)

AIMessage(content='<think>\n好的，用户问“余弦是什么意思？”我需要先回忆一下余弦的基本概念。余弦函数通常用于三角函数中，定义在单位圆上，表示从x轴到角θ的对边与斜边的比例。所以回答时应该简明扼要地解释清楚。\n\n用户可能是在学习数学或者相关领域，需要基础的定义。考虑到字数限制，不能太复杂，所以用20个字或更少的话表达即可。比如“余弦是单位圆上角θ的对边与斜边之比。”这样既准确又简洁。\n\n检查一下是否符合要求，确保不超过20个字。确认没有多余的信息，同时覆盖了定义的关键点。这样用户就能得到清晰、简明的回答。\n</think>\n\n余弦是单位圆上角θ的对边与斜边之比。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 45, 'total_tokens': 224, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen3-0.6b', 'system_fingerprint': 'qwen3-0.6b', 'id': 'chatcmpl-tq26fwjftlh6yw6jjvtqrj', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--49e9352b-f24b-47af-a694-85f0420be55e-0', usage_metadata={'input_tokens': 45, 'output_tokens': 179, 'total_tokens': 224, 'input_token_details': {}, 'output_token_details': {}})

In [4]:
with_message_history.invoke(
    {"ability": "math", "input": "能否再说一遍？"},
    config={"configurable": {"session_id": "abc123"}},
)

AIMessage(content='<think>\n好的，用户之前问了“余弦是什么意思？”我给出了定义，现在他们再次要求再说一遍。我需要确保回答简洁，不超过20个字。\n\n首先，确认用户的需求是重复之前的解释。可能用户希望更清晰或更准确的版本。检查我的回复是否符合要求，有没有超过限制。原句是“余弦是单位圆上角θ的对边与斜边之比。”刚好20个字，没问题。\n\n需要确保没有遗漏关键信息，同时保持简洁。用户可能是学生或学习者，需要快速理解概念。确认回答正确无误后，给出最终回复。\n</think>\n\n余弦是单位圆中角θ的邻边与斜边的比值。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 76, 'total_tokens': 229, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen3-0.6b', 'system_fingerprint': 'qwen3-0.6b', 'id': 'chatcmpl-k5gligdtwslojw988cyctj', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--ba30b7a3-4ab8-4c27-832b-247f1cabd35d-0', usage_metadata={'input_tokens': 76, 'output_tokens': 153, 'total_tokens': 229, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
# New session_id --> does not remember.
with_message_history.invoke(
    {"ability": "math", "input": "能否再说一遍？"},
    config={"configurable": {"session_id": "def234"}},
)

AIMessage(content='<think>\n好的，用户让我再说一遍之前的回答。我需要确保自己完全理解他们的需求。他们可能是在测试我的反应，或者希望确认内容是否正确。作为数学助手，我要保持简洁和准确，同时避免冗长。用户要求20字或更少，所以必须精炼。回顾之前的回复，确认没有遗漏关键信息。最后检查字数，确保符合要求。\n</think>\n\n再说一遍。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 44, 'total_tokens': 133, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen3-0.6b', 'system_fingerprint': 'qwen3-0.6b', 'id': 'chatcmpl-mk5h8wgsbua3jaihgk0skg', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--fca49b8e-bac6-4b0a-a020-797a32ec02d0-0', usage_metadata={'input_tokens': 44, 'output_tokens': 89, 'total_tokens': 133, 'input_token_details': {}, 'output_token_details': {}})

我们可以通过将 ConfigurableFieldSpec 对象列表传递给 history_factory_config 参数来自定义用于跟踪消息历史记录的配置参数。下面，我们使用两个参数： user_id 和 conversation_id 。

In [6]:
from langchain_core.runnables import ConfigurableFieldSpec

store = {}


def get_session_history(user_id: str, conversation_id: str) -> BaseChatMessageHistory:
    if (user_id, conversation_id) not in store:
        store[(user_id, conversation_id)] = ChatMessageHistory()
    return store[(user_id, conversation_id)]


with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="用户的唯一标识符。",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="对话的唯一标识符。",
            default="",
            is_shared=True,
        ),
    ],
)

In [7]:
with_message_history.invoke(
    {"ability": "math", "input": "你好"},
    config={"configurable": {"user_id": "123", "conversation_id": "1"}},
)

AIMessage(content='<think>\n好的，用户发来“你好”，我需要以20个字或更少的字数进行简洁的回应。首先，保持友好和问候的基调是关键。可能的选项有“您好！有什么可以帮助您吗？”或者“你好，请问有什么可以帮到您的。”但要控制在20字以内。比如“您好，请问有什么可以帮到您的。”刚好20个字。需要检查是否符合要求，并确保没有超过限制。\n</think>\n\n您好，请问有什么可以帮到您的？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 41, 'total_tokens': 151, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen3-0.6b', 'system_fingerprint': 'qwen3-0.6b', 'id': 'chatcmpl-cq9hfnbzr166199944xmgk', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--53ca09b0-cd7c-450e-8f11-8f0e550304c2-0', usage_metadata={'input_tokens': 41, 'output_tokens': 110, 'total_tokens': 151, 'input_token_details': {}, 'output_token_details': {}})

## 具有不同签名的可运行实例的示例

上面的可运行程序接受一个字典作为输入并返回一个 BaseMessage。下面我们展示了一些替代方案。

In [8]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel({"output_message":model})


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    output_messages_key="output_message",
)

with_message_history.invoke(
    [HumanMessage(content="什么是二元一次方程")],
    config={"configurable": {"session_id": "baz"}},
)

{'output_message': AIMessage(content='<think>\n嗯，用户问的是“什么是二元一次方程”。首先，我需要确认他们是否了解二元一次方程的基本概念。可能他们是在学习代数或者数学基础，所以需要一个清晰的定义。\n\n接下来，我要回忆一下二元一次方程的标准形式。通常来说，二元一次方程有两个变量，比如x和y，而且每个变量的次数都是1次。也就是说，方程中包含两个未知数，并且所有项的最高次数都是1。\n\n然后，我需要考虑用户可能的背景知识。他们可能已经知道一元一次方程，所以需要区分二元的情况。同时，是否需要举例说明？比如给出一个简单的例子，帮助理解。\n\n还要注意用户的潜在需求。他们可能是在准备考试或者作业，需要明确的定义来应用到解题过程中。因此，解释时要简洁明了，避免使用过于专业的术语，但又要准确。\n\n另外，用户可能对“方程”这个词不太熟悉，所以需要强调二元一次方程是含有两个未知数的一次方程组。同时，是否需要提到解法？比如联立求解的方法？\n\n还要检查是否有其他可能的误解，比如是否混淆了二次方程或者一元一次方程的不同情况。确保定义准确无误。\n\n最后，总结一下，用简短但清晰的语言回答用户的问题，并提供一个明确的例子，帮助他们更好地理解。\n</think>\n\n二元一次方程是指含有两个未知数（通常表示为x和y）且每个未知数的次数都是1次的一元一次方程组。其基本形式是：\n\n$$\nax + by = c\n$$\n\n其中：\n- $a$、$b$、$c$ 是常数；\n- $x$ 和 $y$ 是两个变量。\n\n**特点：**\n1. **两个未知数**：包含两个变量。\n2. **次数为1**：每个变量的最高次数都是1次。\n3. **方程组形式**：通常表示为联立方程，例如：\n   $$\n   \\begin{cases}\n   ax + by = c \\\\\n   dx + ey = f\n   \\end{cases}\n   $$\n\n二元一次方程的核心在于它同时涉及两个变量，并且解法通常是通过代入或消元法求解。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 4

In [9]:
with_message_history.invoke(
    [HumanMessage(content="什么是未知数")],
    config={"configurable": {"session_id": "baz"}},
)

{'output_message': AIMessage(content='<think>\n嗯，用户问的是“什么是未知数”。首先，我需要明确用户的需求是什么。他们可能是在学习数学或者正在准备考试，对方程和变量的概念不太清楚。根据之前的对话历史，用户已经了解过二元一次方程的基本概念，现在可能是在进一步探讨变量的定义。\n\n接下来，我要考虑用户的背景知识。如果用户之前学过代数，那么“未知数”应该指的是解方程时需要求出的变量。但如果是刚开始学习的话，可能需要更基础的解释。因此，在回答时要兼顾不同层次的理解，避免过于深入或模糊。\n\n然后，我需要确保定义准确且清晰。二元一次方程中的未知数是两个变量x和y，而用户的问题可能是在问这些变量的基本含义。同时，要注意区分未知数和常数，因为方程中可能存在多个未知数，但通常只涉及两个。\n\n另外，用户可能希望了解如何应用未知数到实际问题中，或者为什么在解方程时需要考虑未知数的存在。因此，在回答中可以提到未知数的定义、作用以及它们在解题过程中的重要性。\n\n还要检查是否有其他潜在的问题，比如用户是否混淆了变量和系数，或者是否想知道如何表示未知数。不过根据当前问题，保持简洁明了的回答应该是最好的选择。\n</think>\n\n**未知数**是指在数学方程中需要求解的变量，通常用字母（如x、y等）来代表。它们是方程中的未知部分，用于表达未知量之间的关系或条件。\n\n### 举例说明：\n- 在二元一次方程 $2x + 3y = 6$ 中，**x** 和 **y** 是未知数。\n- 解这个方程时，需要找到满足该等式的变量值（即求解未知数）。\n\n### 关键点：\n1. **定义**：未知数是方程中未被确定的量。  \n2. **作用**：它们决定了方程的结构和解题目标。  \n3. **表示方式**：通常用字母或符号表示，如 $x$、$y$ 等。\n\n如果需要进一步解释方程组或解法，请随时提问！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 464, 'prompt_tokens': 216, 'total_tokens': 680, 'completion_tokens_deta

In [10]:
RunnableWithMessageHistory(
    model,
    get_session_history,
)

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x7ba79b89e170>, history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [11]:
from operator import itemgetter

RunnableWithMessageHistory(
    itemgetter("input_messages") | model,
    get_session_history,
    input_messages_key="input_messages",
)

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  input_messages: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x7ba79b89e170>, input_messages_key='input_messages', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

## 持久存储

在许多情况下，最好保留对话历史记录。 RunnableWithMessageHistory 不知道 get_session_history 可调用对象如何检索其聊天消息历史记录。有关使用本地文件系统的示例，请参阅此处。下面我们演示如何使用 Redis。查看内存集成页面，了解使用其他提供程序实现聊天消息历史记录。

如果尚未安装 Redis，我们需要安装它：

pip install --upgrade --quiet redis

如果我们没有可连接的现有 Redis 部署，请启动本地 Redis Stack 服务器：

docker run -d -p 6379:6379 -p 8001:8001 redis/redis-stack:latest

REDIS_URL = "redis://localhost:6379/0"

更新消息历史记录实现只需要我们定义一个新的可调用对象，这次返回 RedisChatMessageHistory 的实例：






In [12]:
from langchain_community.chat_message_histories import RedisChatMessageHistory


def get_message_history(session_id: str) -> RedisChatMessageHistory:
    return RedisChatMessageHistory(session_id, url=REDIS_URL)


with_message_history = RunnableWithMessageHistory(
    runnable,
    get_message_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [13]:
with_message_history.invoke(
    {"ability": "math", "input": "What does cosine mean?"},
    config={"configurable": {"session_id": "foobar"}},
)

NameError: name 'REDIS_URL' is not defined

In [ ]:
with_message_history.invoke(
    {"ability": "math", "input": "What's its inverse"},
    config={"configurable": {"session_id": "foobar"}},
)